# Sequence Length Comparison and Generalisation

Single notebook workflow:
1. Run sequence-length benchmark and save results to disk.
2. Load results from disk.
3. Provide plots and tables comparing performance, latency, and memory usage across different sequence lengths and models.

In [ ]:
from pathlib import Path

from IPython.display import display
from pfns.utils import get_default_device
from pfns.experiments.model_benchmarks.hashing import single_model_hash

from pfns.experiments.model_benchmarks.analysis import (
    add_normalized_comparison_metrics,
    add_numeric_buckets,
    compute_global_normalization_constants,
    compute_mean_rank_tables,
)
from pfns.experiments.model_benchmarks.constants import (
    DEFAULT_BUCKET_BINS,
    DEFAULT_BUCKET_LABELS,
)
from pfns.experiments.model_benchmarks.evaluation import evaluate_models_over_seqlens
from pfns.experiments.model_benchmarks.fixed_batches import resolve_fixed_batches
from pfns.experiments.model_benchmarks.io import (
    SEQ_LEN_REQUIRED_FILES,
    download_results_bundle_from_wandb,
    load_results_bundle,
    make_bundle_path,
    make_model_artifact_name,
    sanitize_wandb_artifact_component,
    save_results_bundle,
    upload_results_bundle_to_wandb,
)
from pfns.experiments.model_benchmarks.model_registry import (
    MODEL_FAMILIES,
    get_autocast_models_from_registry,
    get_forward_models_from_registry,
    get_models_from_families,
    get_models_from_names,
    get_all_models,
)
from pfns.experiments.model_benchmarks.models import load_models_for_benchmark
from pfns.utils import build_exp_outputs_path
from pfns.experiments.model_benchmarks.plotting import build_model_style_map, plot_curves_from_df
from pfns.experiments.model_benchmarks.workflows import (
    alias_single_model_seq_len_bundle,
    build_seq_len_run_metadata,
    merge_seq_len_model_results,
    seq_len_bundle_is_compatible,
    single_model_seq_len_result_from_bundle,
)

EXPERIMENT = {
    "name": "seq_len_comparison",
    "num_classes": 5,
    "num_features": 10,
    "num_test_samples": 100,
    "num_repetitions": 500,
    # "only_numerical_features": True, # default false
    "data_generation_seed": 42,
    "use_warmup_iters": False,
    "print_timing": False,
    "seqlen_list": [250, 500, 750, 1_000, 2_000, 4_000, 8_000, 16_000, 32_000, 64_000, 128_000],
}

WANDB = {
    "enabled": True,
    "artifact_name": "seq_len_comparison",
    "overwrite": False,
    "entity": "icl_arch",
    "project": "seq_len_exp",
    "batch_artifact_name": "seq_len_batches",
    "batch_project": "seq_len_batch_cache",
}

OUTPUT_ROOT = build_exp_outputs_path(Path.cwd(), "seq_len")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print(f"Results are stored in: {OUTPUT_ROOT}")
print(f"Available model families: {list(MODEL_FAMILIES)}")

# Model selection:


# -> All models
# models_to_compare = get_all_models()

# -> All fla models
# models_to_compare = get_models_from_families(["fla_models"])

# -> Equal Param models 
models_to_compare = get_models_from_families(["equal_params_new"])

# -> Transformer masking experiments
# models_to_compare = get_models_from_families(["transformer_masked"])

# models_to_compare = get_models_from_families(["equal_params"])
#models_to_compare = get_models_from_names(["Oracle_Hidden_State_DeltaNet_Comb_ST_new_base", "subsampled:DeltaNet_Comb_ST_3K",  "subsampled:Transformer_Comb_ST_3K","equal_params:Transformer_Comb_ST", "subsampled:DeltaNet_Comb_ST"])

# -> specific family
# models_to_compare = get_models_from_families(["linear_attention"])
# models_to_compare.update(get_models_from_names(["Linear_Attention_Non_Causal", "Linear_Attention_FLA_Comb_ST", "GLA_Comb_ST"]))

# models_to_compare = get_models_from_names([
#     "Oracle_Hidden_State_DeltaNet_Comb_ST_state_init", 
#     "Oracle_Hidden_State_DeltaNet_Comb_ST",
#     "Oracle_Hidden_State_GLA_Comb_ST",
#     "Oracle_Hidden_State_Linear_Attention_Non_Causal",
#     #"Oracle_Hidden_State_Linear_Attention_Non_Causal_new_base",
#     "Linear_Attention_Non_Causal", 
#     # "Softmax_Transformer",
#     ]
# )

# Gating comparision
# models_to_compare = get_models_from_names([
#     "mimetic:GLA_Comb_ST_Ref",
#     "mimetic:GLA_Comb_ST_Open_Gates",
#     "Linear_Attention_Causal_Comb_ST",
#     ]
# )

# State passing comparision
# models_to_compare = get_models_from_names([
#     "State_Passing_GLA_Comb_ST",
#     "GLA_Comb_ST",
#     ]
# )

# models_to_compare = get_models_from_families(["causal_linear_attention_experiments"])
# models_to_compare.update(get_models_from_names(
#     ["Linear_Attention_Non_Causal", "Linear_Attention_FLA_Comb_ST"]
# ))

# TTT models
# models_to_compare = get_models_from_names([
#     "MesaNet_Int_MT",
#     "Linear_Attention_Non_Causal",
#     "Linear_Attention_FLA_Comb_ST",
#     "DeltaNet_Int_MT",
#     "Softmax_Transformer",
#     ]
# )

# models_to_compare = get_models_from_names([
#     # "Linear_Attention_Non_Causal",
#     # "Linear_Attention_Non_Causal_updated",
#     #"DeltaNet_Comb_ST",
#     #"DeltaNet_Comb_ST_no_self_attn",
#     # "Softmax_Transformer"
#     # "Linear_Attention_Non_Causal_with_k_sum_norm",
#     # "Linear_Attention_Non_Causal_fro_norm",
#     # "Linear_Attention_Non_Causal_fro_norm_from_causal",
#     # "Linear_Attention_Non_Causal_with_k_sum_norm",
#     # "Linear_Attention_Causal_fro_norm_from_non_causal",
#     # "Linear_Attention_Comb_ST",
#     # "Linear_Attention_Comb_ST_old_setup",
#     # "Linear_Attention_Comb_ST_fro_norm",
#     # "Linear_Attention_FLA_Comb_ST",
#     #"Linear_Attention_FLA_Comb_ST_wo_self_term",
#     #"DeltaNet_Comb_ST_Seq_Len_200-100K_loguniform",
#     # "Transformer_Non_Causal",
#     "masked:Transformer_Comb_ST",
#     # "Transformer_Comb_MT"
#     "Linear_Attention_Non_Causal",
#     "Linear_Attention_Comb_ST",
# ])

# models_to_compare.update(get_models_from_families(["oracles"]))


# models_to_compare = get_models_from_families(["deltanet_high_seq_len"])

# MODEL_NAMES = [
#     'Linear_Attention_Non_Causal',
#     'Linear_Attention_Comb_ST',
#     'Linear_Attention_Non_Causal_fro_norm',
#     'Linear_Attention_Comb_ST_fro_norm',
# ]

# models_to_compare = get_models_from_names(MODEL_NAMES)

# models_to_compare = get_models_from_families(["bidirectional"])

# models_to_compare.update(get_models_from_names([
#     "Softmax_Transformer",
# ]))

device = get_default_device()
print(f"Using device: {device}")

TASK_KWARGS = (
    {"only_numerical_features": True}
    if EXPERIMENT.get("only_numerical_features", False)
    else {}
)

expected_run_metadata = build_seq_len_run_metadata(experiment=EXPERIMENT, device=device)

precomputed_batches = resolve_fixed_batches(
    experiment=EXPERIMENT,
    output_root=OUTPUT_ROOT,
    default_device=device,
    wandb=WANDB,
    task_kwargs=TASK_KWARGS,
)
print(f"Using {len(precomputed_batches)} fixed batches for this experiment.")


## Running the sequence-length benchmark

In [ ]:
results_by_model = {}
model_bundle_paths = {}

if WANDB["enabled"] and WANDB["overwrite"]:
    print("WANDB overwrite=True: skipping per-model download and forcing rerun.")


for model_name, model_config in models_to_compare.items():
    model_hash = single_model_hash(
        model_name=model_name,
        model_config=model_config,
        experiment_payload=EXPERIMENT,
    )
    model_artifact_name = make_model_artifact_name(
        base_artifact_name=WANDB["artifact_name"],
        model_name=model_name,
        model_hash=model_hash,
    )

    reused_cached_result = False
    if WANDB["enabled"] and not WANDB["overwrite"]:
        cached_bundle_path = download_results_bundle_from_wandb(
            artifact_name=model_artifact_name,
            entity=WANDB["entity"],
            project=WANDB["project"],
            download_root=OUTPUT_ROOT / "wandb_model_cache",
            required_files=SEQ_LEN_REQUIRED_FILES,
        )
        if cached_bundle_path is not None:
            cached_bundle = load_results_bundle(cached_bundle_path)
            cached_bundle_for_model, aliased_from = alias_single_model_seq_len_bundle(
                cached_bundle,
                target_model_name=model_name,
            )
            if seq_len_bundle_is_compatible(
                cached_bundle_for_model,
                model_name=model_name,
                expected_metadata=expected_run_metadata,
            ):
                results_by_model[model_name] = single_model_seq_len_result_from_bundle(
                    cached_bundle_for_model,
                    model_name=model_name,
                )
                model_bundle_paths[model_name] = cached_bundle_path
                reused_cached_result = True
                if aliased_from is not None and aliased_from != model_name:
                    print(
                        f"Reused cached W&B result for {model_name} from stored label "
                        f"{aliased_from}: {cached_bundle_path}"
                    )
                else:
                    print(f"Reused cached W&B result for {model_name}: {cached_bundle_path}")
            else:
                print(
                    f"Cached artifact for {model_name} is incompatible with "
                    "this run metadata. Rerunning model."
                )

    if reused_cached_result:
        continue

    print(f"Running benchmark for model: {model_name}")
    models, configs = load_models_for_benchmark({model_name: model_config}, device=device)
    model_results = evaluate_models_over_seqlens(
        models=models,
        configs=configs,
        seqlen_list=EXPERIMENT["seqlen_list"],
        num_features=EXPERIMENT["num_features"],
        num_classes=EXPERIMENT["num_classes"],
        number_of_test_samples=EXPERIMENT["num_test_samples"],
        number_of_repetitions=EXPERIMENT["num_repetitions"],
        use_warmup_iters=EXPERIMENT["use_warmup_iters"],
        print_timing=EXPERIMENT["print_timing"],
        autocast_models=get_autocast_models_from_registry(
            {model_name: model_config},
            device=device,
        ),
        device=device,
        data_generation_seed=EXPERIMENT["data_generation_seed"],
        subsample_dataset_size=model_config.get("subsample_dataset_size"),
        progress_desc=f"{model_name} progress",
        forward_models=get_forward_models_from_registry({model_name: model_config}),
        task_kwargs=TASK_KWARGS,
        precomputed_batches=precomputed_batches,
    )
    results_by_model[model_name] = model_results

    model_bundle_path = make_bundle_path(
        OUTPUT_ROOT,
        f"{EXPERIMENT['name']}_{sanitize_wandb_artifact_component(model_name)}",
    )
    save_results_bundle(
        model_results,
        model_bundle_path,
        experiment=EXPERIMENT,
        include_raw_torch=True,
    )
    model_bundle_paths[model_name] = model_bundle_path
    print(f"Saved per-model bundle for {model_name}: {model_bundle_path}")

    if WANDB["enabled"]:
        artifact_ref = upload_results_bundle_to_wandb(
            model_bundle_path,
            artifact_name=model_artifact_name,
            entity=WANDB["entity"],
            project=WANDB["project"],
            run_name=(
                f"{EXPERIMENT['name']}_{sanitize_wandb_artifact_component(model_name)}_"
                f"{model_hash}"
            ),
            metadata={
                "experiment": EXPERIMENT,
                "model_name": model_name,
                "model_config": model_config,
                "model_hash": model_hash,
                "run_metadata": model_results.get("metadata", {}),
            },
        )
        print(f"Uploaded per-model artifact for {model_name}: {artifact_ref}")

merged_output = merge_seq_len_model_results(
    results_by_model=results_by_model,
    expected_run_metadata=expected_run_metadata,
    model_names=list(models_to_compare.keys()),
    experiment=EXPERIMENT,
    model_bundle_paths=model_bundle_paths,
)

results = merged_output["results"]
metric_df = merged_output["metric_df"]
timing_df = merged_output["timing_df"]
memory_df = merged_output["memory_df"]
bundle_metadata = merged_output["bundle_metadata"]


## Check benchmark results

In [ ]:
if "results" not in globals():
    raise RuntimeError("No in-memory merged results found. Run the benchmark cell first.")

print("Using in-memory merged results from per-model artifacts/runs.")

print("Bundle metadata:")
display(bundle_metadata)

print("metric_df shape:", metric_df.shape)
print("timing_df shape:", timing_df.shape)
print("memory_df shape:", memory_df.shape)

display(metric_df.head())


# Performance, latency and memory curves.

In [ ]:
import numpy as np
import pandas as pd

BASE_METRIC_LABELS = {
    # "acc": "Accuracy",
    # "ce": "Cross Entropy",
    "roc_auc": "ROC AUC",
}
HIGHER_IS_BETTER_METRICS = {"acc", "roc_auc"}
GLOBAL_NORMALIZATION_REFERENCE_FAMILIES = ["equal_params", "oracles"] # ["equal_params", "oracles"] #["deltanet_high_seq_len"] #
GLOBAL_NORMALIZATION_REFERENCE_MODEL_NAMES = []
GLOBAL_NORMALIZATION_LOWER_BOUND_REFERENCE_MAX_SEQLEN = 1_000
NORMALIZATION_VIEWS = {
    "metric_normalized_per_seqlen": {
        "prefix": "normalized_",
        "title": "Normalised Across Models at Fixed SeqLen",
        "group_cols": ("seqlen", "rep"),
        "normalization_scope": "comparison",
    },
    "metric_normalized_global": {
        "prefix": "normalized_global_",
        "title": "Normalised ROC AUC Across Models and Sequence Length",
        "group_cols": ("rep",),
        "normalization_scope": "group",
        "require_normalization_constants": True,
    },
    "metric_normalized_per_model": {
        "prefix": "normalized_per_model_",
        "title": "Normalised Across SeqLen Within Model",
        "group_cols": ("model", "rep"),
        "normalization_scope": "group",
    },
}

SEQ_LEN_ANALYSIS_MAX = None  # keep only seqlen <= cutoff for all downstream analyses.
SEQ_LEN_ANALYSIS_DROP_LOWEST = False
SEQ_LEN_ANALYSIS_MIN = (
    sorted(EXPERIMENT["seqlen_list"])[1]
    if SEQ_LEN_ANALYSIS_DROP_LOWEST and len(EXPERIMENT["seqlen_list"]) > 1
    else None
)

PLOT_DISPLAY_NAME_OVERRIDES = {
    "Softmax_Transformer": "Softmax Non-Causal",
    "Transformer_Non_Causal": "Softmax Non-Causal",
    "equal_params:Transformer_Comb_ST": "Softmax Non-Causal",
    "Linear_Attention_Non_Causal": "Linear Non-Causal",
    "DeltaNet_Comb_ST": "DeltaNet",
    "DeltaNet_Comb_ST_Reference_New": "DeltaNet",
    "oracles:DeltaNet_Comb_ST": "DeltaNet",
}

def _filter_to_analysis_seqlen_range(
    df,
    *,
    min_seqlen=SEQ_LEN_ANALYSIS_MIN,
    max_seqlen=SEQ_LEN_ANALYSIS_MAX,
):
    if df is None:
        return df
    filtered = df.copy()
    if min_seqlen is not None:
        filtered = filtered[filtered["seqlen"] >= int(min_seqlen)].copy()
    if max_seqlen is not None:
        filtered = filtered[filtered["seqlen"] <= int(max_seqlen)].copy()
    return filtered

def _apply_plot_display_name_overrides(df):
    if df is None:
        return df
    plot_df = df.copy()
    if "display_name" not in plot_df.columns:
        plot_df["display_name"] = pd.NA
    for model_name, display_name in PLOT_DISPLAY_NAME_OVERRIDES.items():
        plot_df.loc[plot_df["model"].astype(str) == model_name, "display_name"] = display_name
    return plot_df

metric_analysis_df = _apply_plot_display_name_overrides(_filter_to_analysis_seqlen_range(metric_df))
timing_analysis_df = _apply_plot_display_name_overrides(_filter_to_analysis_seqlen_range(timing_df))
memory_analysis_df = _apply_plot_display_name_overrides(_filter_to_analysis_seqlen_range(memory_df))

if SEQ_LEN_ANALYSIS_MIN is None and SEQ_LEN_ANALYSIS_MAX is None:
    print("Using all sequence lengths for downstream analyses.")
else:
    bounds = []
    if SEQ_LEN_ANALYSIS_MIN is not None:
        bounds.append(f"seqlen >= {int(SEQ_LEN_ANALYSIS_MIN)}")
    if SEQ_LEN_ANALYSIS_MAX is not None:
        bounds.append(f"seqlen <= {int(SEQ_LEN_ANALYSIS_MAX)}")
    print(f"Filtering downstream analyses to {' and '.join(bounds)}.")



In [ ]:
# This cell calculates the global normalization constants based on the specified reference models
def load_reference_result(model_name, model_config):
    if model_name in globals().get("results_by_model", {}):
        return results_by_model[model_name]
    if not WANDB["enabled"]:
        raise RuntimeError(f"Reference model {model_name!r} is not in memory and W&B is disabled.")

    model_hash = single_model_hash(model_name=model_name, model_config=model_config, experiment_payload=EXPERIMENT)
    cached_bundle_path = download_results_bundle_from_wandb(
        artifact_name=make_model_artifact_name(base_artifact_name=WANDB["artifact_name"], model_name=model_name, model_hash=model_hash),
        entity=WANDB["entity"],
        project=WANDB["project"],
        download_root=OUTPUT_ROOT / "wandb_model_cache",
        required_files=SEQ_LEN_REQUIRED_FILES,
    )
    if cached_bundle_path is None:
        raise RuntimeError(f"No cached W&B result found for reference model {model_name!r}.")

    bundle, _ = alias_single_model_seq_len_bundle(load_results_bundle(cached_bundle_path), target_model_name=model_name)
    if not seq_len_bundle_is_compatible(bundle, model_name=model_name, expected_metadata=expected_run_metadata):
        raise RuntimeError(f"Cached W&B result for reference model {model_name!r} has incompatible metadata.")
    return single_model_seq_len_result_from_bundle(bundle, model_name=model_name)

def load_global_normalization_reference_metric_df(model_configs):
    merged_reference = merge_seq_len_model_results(
        results_by_model={name: load_reference_result(name, config) for name, config in model_configs.items()},
        expected_run_metadata=expected_run_metadata,
        model_names=list(model_configs),
        experiment=EXPERIMENT,
    )
    return _apply_plot_display_name_overrides(
        _filter_to_analysis_seqlen_range(merged_reference["metric_df"])
    )

global_normalization_reference_models = {}
if GLOBAL_NORMALIZATION_REFERENCE_FAMILIES:
    global_normalization_reference_models.update(
        get_models_from_families(GLOBAL_NORMALIZATION_REFERENCE_FAMILIES)
    )
if GLOBAL_NORMALIZATION_REFERENCE_MODEL_NAMES:
    global_normalization_reference_models.update(
        get_models_from_names(GLOBAL_NORMALIZATION_REFERENCE_MODEL_NAMES)
    )

global_normalization_reference_df = metric_analysis_df
if global_normalization_reference_models:
    global_normalization_reference_df = load_global_normalization_reference_metric_df(
        global_normalization_reference_models
    )

GLOBAL_NORMALIZATION_CONSTANTS = compute_global_normalization_constants(
    global_normalization_reference_df,
    metric_keys=BASE_METRIC_LABELS,
    higher_is_better_metrics=HIGHER_IS_BETTER_METRICS,
    group_cols=("rep",),
    lower_bound_reference_max_x=GLOBAL_NORMALIZATION_LOWER_BOUND_REFERENCE_MAX_SEQLEN,
)
print(f"Global normalization reference models: {sorted(global_normalization_reference_df['model'].astype(str).unique())}")
print(f"Global normalization constants: {GLOBAL_NORMALIZATION_CONSTANTS}")

NORMALIZATION_VIEWS["metric_normalized_global"]["normalization_constants"] = GLOBAL_NORMALIZATION_CONSTANTS

In [ ]:
def build_normalized_metric_plot_df(df):
    plot_df = df.copy()
    for config in NORMALIZATION_VIEWS.values():
        plot_df = add_normalized_comparison_metrics(
            plot_df,
            metric_keys={
                metric: (f"{config['prefix']}{metric}", f"{config['title']}: {label}")
                for metric, label in BASE_METRIC_LABELS.items()
            },
            higher_is_better_metrics=HIGHER_IS_BETTER_METRICS,
            group_cols=config["group_cols"],
            normalization_scope=config["normalization_scope"],
            normalization_constants=config.get("normalization_constants"),
            require_normalization_constants=config.get("require_normalization_constants", False),
            normalized_prefix=config["prefix"],
        )
    return plot_df

metric_plot_df = build_normalized_metric_plot_df(metric_analysis_df)

NORMALIZED_HIGHER_IS_BETTER = {
    f"{config['prefix']}{metric}"
    for config in NORMALIZATION_VIEWS.values()
    for metric in HIGHER_IS_BETTER_METRICS
}

SEQ_LEN_GENERALIZATION_PLOT_MODE = "individual_runs"
SEQ_LEN_GENERALIZATION_RUN_ALPHA = 0.07
SEQ_LEN_GENERALIZATION_DISTRIBUTION_ALPHA = 0.3
SEQ_LEN_GENERALIZATION_DISTRIBUTION_WIDTH_FRAC = 0.4
PLOT_SPECS = {
    "metric_raw": list(BASE_METRIC_LABELS.items()),
    **{
        panel_name: [
            (f"{config['prefix']}{metric}", label)
            for metric, label in BASE_METRIC_LABELS.items()
        ]
        for panel_name, config in NORMALIZATION_VIEWS.items()
    },
    "metric_rank": list(BASE_METRIC_LABELS.items()),
    "timing": [
        # ("forward_time_ms", "Total Time (ms)"),
        ("fit_time_ms", "Fit Time (ms)"),
        ("predict_time_ms", "Predict Time (ms)"),
    ],
    "memory": [
        # ("peak_allocated_mb", "Peak Allocated MB"),
        # ("peak_reserved_mb", "Peak Reserved MB"),
        ("context_size_mb", "Context Size MB"),
    ],
}

PAPER_PLOT_FIGSIZE = (8.0, 4.8)

PLOT_PANEL_CONFIGS = [ # error_style can be "bars" or "band"
    ("metric_raw", metric_plot_df, 
        {"log_x": True, 
         "log_y": False, 
        #  "plot_mode": SEQ_LEN_GENERALIZATION_PLOT_MODE, 
        #  "run_alpha": SEQ_LEN_GENERALIZATION_RUN_ALPHA, 
        #  "distribution_alpha": SEQ_LEN_GENERALIZATION_DISTRIBUTION_ALPHA,
        #  "distribution_width_frac": SEQ_LEN_GENERALIZATION_DISTRIBUTION_WIDTH_FRAC,
        #  "show_run_lines": True,
         "distribution_style": "none", #"half_violin",
         "title_suffix": " vs Number of In-context Samples",
         "figsize": PAPER_PLOT_FIGSIZE,
        # "error_bars": "ci95", "error_style": "band",
        }
    ),
    *[
        (
            panel_name,
            metric_plot_df,
            {
                "log_x": True,
                "log_y": False,
                # "plot_mode": SEQ_LEN_GENERALIZATION_PLOT_MODE,
                # "run_alpha": SEQ_LEN_GENERALIZATION_RUN_ALPHA,
                "distribution_alpha": SEQ_LEN_GENERALIZATION_DISTRIBUTION_ALPHA,
                "distribution_width_frac": SEQ_LEN_GENERALIZATION_DISTRIBUTION_WIDTH_FRAC,
                "show_run_lines": False,
                "distribution_style": "half_violin",
                "title_suffix": f" {config['title']}",
                "title_override": "" if panel_name == "metric_normalized_global" else None,
                "value_ylim": (0.3, 0.82) if panel_name == "metric_normalized_global" else (-0.05, 1.05),
                "y_label": "Normalised ROC AUC" if panel_name == "metric_normalized_global" else None,
                "figsize": PAPER_PLOT_FIGSIZE,
                "error_bars": "ci95", "error_style": "band",
            },
        )
        for panel_name, config in NORMALIZATION_VIEWS.items()
    ],
    ("timing", timing_analysis_df, {"log_x": True, "log_y": True, "log_y_metrics": [], "show_std": False, "seconds": ["forward_time_ms", "fit_time_ms"], "title_suffix": " vs Number of In-context Samples", "panel_wspace": 0.18, "legend_y": 0.0}),
    ("memory", memory_analysis_df, {"log_x": True, "log_y": True, "gigabytes": False, "title_suffix": " vs Number of In-context Samples", "panel_wspace": 0.28, "legend_y": 0.0},),
]

MODEL_LEGEND_LAYOUT = "bottom" # bottom or right

TABLE_PANEL_CONFIGS = [
    ("Performance", metric_plot_df, {"value_fmt": "{:.4f}", "cmap": "Greens_r"}),
    ("Timing", timing_analysis_df, {}),
    ("Memory", memory_analysis_df, {}),
]

def _prepare_plot_panel(df, specs, *, seconds=False, gigabytes=False):
    plot_df = df.copy()
    plot_specs = list(specs)
    seconds_metrics = {metric for metric, _ in plot_specs} if seconds is True else set(seconds or [])
    if seconds_metrics:
        plot_df = plot_df.copy()
        mask = plot_df["metric"].astype(str).isin(seconds_metrics)
        plot_df.loc[mask, "value"] = plot_df.loc[mask, "value"] / 1000.0
        plot_specs = [
            (metric, label.replace("(ms)", "(s)")) if metric in seconds_metrics else (metric, label)
            for metric, label in plot_specs
        ]
    if gigabytes:
        plot_df = plot_df.copy()
        plot_df["value"] = plot_df["value"] / 1024.0
        plot_specs = [(metric, label.replace("MB", "GB")) for metric, label in plot_specs]
    return plot_df, plot_specs

def plot_panel(
        df, specs, *, value_col="value", show_std=False, error_bars=None, error_style="bars", 
        plot_mode="aggregate", run_alpha=0.35, distribution_alpha=None, distribution_width_frac=0.18, 
        show_run_lines=True, distribution_style="half_violin", log_x=False, log_y=False, invert_y=False, 
        seconds=False, gigabytes=False, title_suffix="", title_override=None, model_legend_layout=MODEL_LEGEND_LAYOUT, 
        value_ylim=None, y_label=None, log_y_metrics=None, panel_wspace=None, legend_y=None, figsize=None, style_map_override=None, show=True
    ):
    plot_df, plot_specs = _prepare_plot_panel(df, specs, seconds=seconds, gigabytes=gigabytes)
    fig, axes = plot_curves_from_df(
        plot_df,
        specs=plot_specs,
        style_map=style_map if style_map_override is None else style_map_override,
        x_col="seqlen",
        value_col=value_col,
        x_label="Number of In-context Samples",
        title_suffix=title_suffix,
        show_std=show_std,
        error_bars=error_bars,
        error_style=error_style,
        plot_mode=plot_mode,
        run_alpha=run_alpha,
        distribution_alpha=distribution_alpha,
        distribution_width_frac=distribution_width_frac,
        show_run_lines=show_run_lines,
        distribution_style=distribution_style,
        log_x=log_x,
        log_y=log_y,
        invert_y=invert_y,
        model_legend_layout=model_legend_layout,
        figsize=figsize,
        show=show,
    )
    if log_y_metrics is not None and axes is not None:
        log_y_metric_set = set(log_y_metrics)
        for ax, (metric_key, _) in zip(axes, plot_specs):
            if metric_key in log_y_metric_set:
                ax.set_yscale("log")
    if panel_wspace is not None and fig is not None:
        fig.subplots_adjust(wspace=float(panel_wspace))
    if legend_y is not None and fig is not None:
        for legend in fig.legends:
            legend.set_bbox_to_anchor((0.5, float(legend_y)), transform=fig.transFigure)
    if value_ylim is not None and axes is not None:
        for ax in axes:
            ax.set_ylim(*value_ylim)
    if y_label is not None and axes is not None:
        for ax in axes:
            ax.set_ylabel(y_label)
    if title_override is not None and axes is not None:
        for ax in axes:
            ax.set_title(title_override)
    return fig, axes

def _display_pivot_by_metric(df, *, panel_name: str, pivot_col: str, value_col: str = "value", value_fmt: str = "{:.3f}", cmap: str = "Blues_r"):
    if df.empty:
        print(f"No {panel_name} data available.")
        return

    for metric in sorted(df["metric"].unique()):
        print(f"{panel_name} | {metric}")
        table = df[df["metric"] == metric].pivot_table(
            index="model",
            columns=pivot_col,
            values=value_col,
            observed=True,
        )
        table = order_model_rows(table)
        display(table.style.format(value_fmt).background_gradient(cmap=cmap, axis=None))

def display_panel_tables(df, *, panel_name: str, value_fmt: str = "{:.3f}", cmap: str = "Blues_r"):
    if df.empty:
        print(f"No {panel_name} data available.")
        return

    print(f"=== {panel_name}: Mean by model ===")
    overall = (
        df.groupby(["model", "metric"], observed=True)["value"]
        .mean()
        .reset_index()
        .pivot_table(index="model", columns="metric", values="value", observed=True)
    )
    overall = order_model_rows(overall)
    display(overall.style.format(value_fmt).background_gradient(cmap=cmap, axis=None))

    bucket_df = add_numeric_buckets(
        df,
        value_col="seqlen",
        bucket_col="bucket",
        bins=DEFAULT_BUCKET_BINS,
        labels=DEFAULT_BUCKET_LABELS,
    )
    print(f"=== {panel_name}: Mean by sequence-length bucket ===")
    _display_pivot_by_metric(bucket_df, panel_name=panel_name, pivot_col="bucket", value_col="value", value_fmt=value_fmt, cmap=cmap)

    print(f"=== {panel_name}: Mean by sequence length ===")
    _display_pivot_by_metric(df, panel_name=panel_name, pivot_col="seqlen", value_col="value", value_fmt=value_fmt, cmap=cmap)

DELTANET_HIGH_SEQ_LEN_MODEL_ORDER = [
    "DeltaNet_Comb_ST_Seq_Len_200-100K_loguniform",
    "DeltaNet_Comb_ST_Seq_Len_200-64K_loguniform",
    "DeltaNet_Comb_ST_Seq_Len_200-32K_loguniform",
    "DeltaNet_Comb_ST_Seq_Len_200-16K_loguniform",
    "DeltaNet_Comb_ST_Seq_Len_200-8K_loguniform",
    "DeltaNet_Comb_ST_Reference",
]

MODEL_PLOT_ORDER = [
    "Softmax_Transformer",
    "Transformer_Non_Causal",
    "equal_params:Transformer_Comb_ST",
    "masked:Transformer_Comb_ST",
    "Linear_Attention_Non_Causal",
    "Linear_Attention_Comb_ST",
    "DeltaNet_Comb_ST_Reference_New",
    "DeltaNet_Comb_ST",
    "oracles:DeltaNet_Comb_ST",
    *DELTANET_HIGH_SEQ_LEN_MODEL_ORDER,
    "oracles:Oracle_Hidden_State_DeltaNet_Comb_ST",
    "oracles:subsampled:DeltaNet_Comb_ST_3K",
]

DELTANET_HIGH_SEQ_LEN_STYLE_OVERRIDES = {
    "DeltaNet_Comb_ST_Seq_Len_200-100K_loguniform": ("X", "-", "#FDBB84"),
    "DeltaNet_Comb_ST_Seq_Len_200-64K_loguniform": ("^", "-", "#FC8D59"),
    "DeltaNet_Comb_ST_Seq_Len_200-32K_loguniform": ("D", "-", "#EF6548"),
    "DeltaNet_Comb_ST_Seq_Len_200-16K_loguniform": ("s", "-", "#D7301F"),
    "DeltaNet_Comb_ST_Seq_Len_200-8K_loguniform": ("v", "-", "#A50F15"),
    "DeltaNet_Comb_ST_Reference": ("o", "-", "#67000D"),
}

SEQ_LEN_STYLE_OVERRIDES = {
    "Softmax_Transformer": ("s", "-", "#08519C"),
    "Transformer_Non_Causal": ("s", "-", "#08519C"),
    "equal_params:Transformer_Comb_ST": ("s", "-", "#08519C"),
    "masked:Transformer_Comb_ST": ("D", "--", "#3182BD"),
    "Linear_Attention_Non_Causal": ("<", "-", "#00796B"),
    "Linear_Attention_Comb_ST": (">", "--", "#26A69A"),
    "oracles:Oracle_Hidden_State_DeltaNet_Comb_ST": ("o", ":", "#000000"),
    "DeltaNet_Comb_ST": ("v", "-", "#B2182B"),
    "DeltaNet_Comb_ST_Reference_New": ("v", "-", "#B2182B"),
    "oracles:DeltaNet_Comb_ST": ("v", "-", "#B2182B"),
    **DELTANET_HIGH_SEQ_LEN_STYLE_OVERRIDES,
    "DeltaNet_Comb_ST_Seq_Len_200-64K_lognormal_dataset_matched": ("X", "-.", "#D65A65"),
    "oracles:subsampled:DeltaNet_Comb_ST_3K": ("^", "--", "#D73027"),
    "State_Passing_GLA_Comb_ST": ("h", ":", "#74C476"),
    "State_Weaving_GLA_Comb_ST": (">", (0, (1, 1)), "#31A354"),
    "mimetic:GLA_Comb_ST_Ref": ("<", "-", "#00441B"),
    "subsampled:GLA_Comb_ST_3K": ("X", ":", "#238B45"),
    "mimetic:GLA_Comb_ST_mimetic_gate_only": (">", ":", "#66A61E"),
    "mimetic:GLA_Comb_ST_mimetic_full": ("p", ":", "#006D2C"),
    "State_Passing_DeltaNet_Comb_ST": ("h", ":", "#FC9272"),
    "Bidirectional_DeltaNet_Comb_ST_mean_output_mean_cache": ("P", ":", "#EF6548"),
    "Bidirectional_GLA_Comb_ST_mean_output_mean_cache": ("8", ":", "#238B45"),
}

def apply_seq_len_style_overrides(style_map):
    style_map.update(
        {
            model: style
            for model, style in SEQ_LEN_STYLE_OVERRIDES.items()
            if model in style_map
        }
    )
    return style_map

present_models = metric_plot_df["model"].astype(str).unique().tolist()
present_model_set = set(present_models)
ordered_models = [name for name in MODEL_PLOT_ORDER if name in present_model_set]
ordered_models.extend(name for name in present_models if name not in ordered_models)

def order_model_rows(table):
    model_order = [model for model in ordered_models if model in table.index]
    model_order.extend(model for model in table.index if model not in model_order)
    return table.reindex(model_order)

style_map = apply_seq_len_style_overrides(build_model_style_map(ordered_models))

def resolve_graphics_dir():
    for base in (Path.cwd(), *Path.cwd().parents):
        candidate = base / "PFNs" / "graphics"
        if candidate.exists():
            return candidate
    return Path.cwd() / "PFNs" / "graphics"

GRAPHICS_DIR = resolve_graphics_dir()
GRAPHICS_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_FORMATS = ("pdf",)

def save_figure(fig, stem, *, formats=FIGURE_FORMATS):
    if fig is None:
        return []
    saved_paths = []
    for fmt in formats:
        path = GRAPHICS_DIR / f"{stem}.{fmt}"
        save_kwargs = {"bbox_inches": "tight"}
        if fmt.lower() in {"png", "jpg", "jpeg"}:
            save_kwargs["dpi"] = 300
        fig.savefig(path, **save_kwargs)
        saved_paths.append(path)
    return saved_paths

saved_figure_paths = []
for panel_name, df, opts in PLOT_PANEL_CONFIGS:
    fig, axes = plot_panel(df, PLOT_SPECS[panel_name], show=False, **opts)
    saved_figure_paths.extend(save_figure(fig, f"seq_len_{panel_name}"))
    if fig is not None:
        display(fig)

print("Saved figures:")
for path in saved_figure_paths:
    print(f" - {path}")


In [ ]:
rank_tables = compute_mean_rank_tables(
    metric_plot_df,
    x_col="seqlen",
    higher_is_better_metrics={"acc", "roc_auc", *NORMALIZED_HIGHER_IS_BETTER},
)

fig, axes = plot_panel(
    rank_tables["x_ranks"],
    PLOT_SPECS["metric_rank"],
    value_col="rank",
    log_x=True,
    invert_y=True,
    title_suffix=" Rank (1 is best)",
    figsize=PAPER_PLOT_FIGSIZE,
    show=False,
)
saved_rank_paths = save_figure(fig, "seq_len_metric_rank")
if fig is not None:
    display(fig)
print("Saved rank figures:")
for path in saved_rank_paths:
    print(f" - {path}")

## Side by side plot of Length Generalisation Gap and Mitigation Attempts

In [ ]:
from pfns.experiments.model_benchmarks.analysis import (
    add_normalized_comparison_metrics,
    compute_global_normalization_constants,
)
from pfns.experiments.model_benchmarks.comparison_analysis import ci95_halfwidth
from pfns.experiments.model_benchmarks.plotting import (
    add_pretraining_split_legend,
    apply_pretraining_split_background,
    build_model_legend_name_map,
)
import matplotlib.lines as mlines
import matplotlib.pyplot as plt

SIDE_BY_SIDE_MODEL_GROUPS = {
    "Length Generalisation Gap": get_models_from_names([
        "equal_params:Transformer_Comb_ST", # Non-causal
        "masked:Transformer_Comb_ST",
        "Linear_Attention_Non_Causal",
        "Linear_Attention_Comb_ST",
        # "oracles:DeltaNet_Comb_ST",
        # "oracles:subsampled:DeltaNet_Comb_ST_3K",
        "mimetic:GLA_Comb_ST_Ref",
        "subsampled:GLA_Comb_ST_3K",
        "oracles:Oracle_Hidden_State_DeltaNet_Comb_ST",
    ]),
    "Mitigation Strategies": get_models_from_names([
        "State_Passing_GLA_Comb_ST",
        "State_Weaving_GLA_Comb_ST",
        "mimetic:GLA_Comb_ST_Ref",
        "mimetic:GLA_Comb_ST_mimetic_gate_only",
        "Bidirectional_GLA_Comb_ST_mean_output_mean_cache",
        # "State_Passing_DeltaNet_Comb_ST",
        # "State_Weaving_DeltaNet_Comb_ST",
        # "Bidirectional_DeltaNet_Comb_ST_mean_output_mean_cache",
        # "oracles:DeltaNet_Comb_ST",
        # "DeltaNet_Comb_ST_Seq_Len_200-64K_lognormal_dataset_matched",
    ]),
}
# SIDE_BY_SIDE_MODEL_GROUPS["Length Generalisation Gap"].update(get_models_from_families(["oracles"]))

# subsampled:GLA_Comb_ST_3K

SIDE_BY_SIDE_STYLE_ORDER = [
    "Transformer_Non_Causal",
    "masked:Transformer_Comb_ST",
    "Linear_Attention_Non_Causal",
    "Linear_Attention_Comb_ST",
    "oracles:DeltaNet_Comb_ST",
    "DeltaNet_Comb_ST_Seq_Len_200-64K_lognormal_dataset_matched",
    "State_Passing_DeltaNet_Comb_ST",
    "State_Weaving_DeltaNet_Comb_ST",
    "Bidirectional_DeltaNet_Comb_ST_mean_output_mean_cache",
    "oracles:subsampled:DeltaNet_Comb_ST_3K",
    "oracles:Oracle_Hidden_State_DeltaNet_Comb_ST",
    "mimetic:GLA_Comb_ST_Ref",
    "State_Passing_GLA_Comb_ST",
    "State_Weaving_GLA_Comb_ST",
    "mimetic:GLA_Comb_ST_mimetic_gate_only",
    "Bidirectional_GLA_Comb_ST_mean_output_mean_cache",
]
SIDE_BY_SIDE_LABEL_OVERRIDES = {
    "Oracle Hidden State": "Hidden State Oracle",
    "Softmax Attention\nNon-Causal": "Softmax Non-Causal",
    "Softmax Attention\nCausal": "Softmax Causal",
    "Linear Attention\nNon-Causal": "Linear Non-Causal",
    "Linear Attention\nCausal": "Linear Causal",
    "DeltaNet\n(Subsampled 3K)": "Delta: Subsampled 3K",
    "DeltaNet with\nState Passing": "Delta: State Passing",
    "DeltaNet with\nWeaving Passing": "Delta: State Weaving",
    "GLA with\nState Passing": "Linear Gated: State Passing",
    "GLA with\nState Weaving": "Linear Gated: State Weaving",
    "GLA with Mimetic": "Linear Gated: Mimetic",
    "GLA Comb ST\n(Subsampled 3K)": "Linear Gated: Subsampled 3K",
    "Bidirectional DeltaNet": "Delta: Bidirectional",
    "Bidirectional GLA ": "Linear Gated: Bidirectional",
    "DeltaNet High Seq Len\ncompute matched": "Delta: Long Sequence",
    "DeltaNet with\nState Weaving": "Delta: State Weaving",
    "GLA": "Linear Gated",
    "DeltaNet": "Delta",
}

SIDE_BY_SIDE_METRIC = "normalized_global_roc_auc"
SIDE_BY_SIDE_VALUE_YLIM = (0.3, 0.82)
SIDE_BY_SIDE_FIGSIZE = (13.5, 5.75)
SIDE_BY_SIDE_CI_ALPHA = 0.055
SIDE_BY_SIDE_DIRECT_LINE_LABELS = {"Hidden State Oracle"}

side_by_side_metric_dfs = {
    title: build_normalized_metric_plot_df(
        load_global_normalization_reference_metric_df(model_configs)
    )
    for title, model_configs in SIDE_BY_SIDE_MODEL_GROUPS.items()
}
side_by_side_combined_df = pd.concat(side_by_side_metric_dfs.values(), ignore_index=True)
side_by_side_display_names = build_model_legend_name_map(side_by_side_combined_df)


def side_by_side_label(model):
    label = side_by_side_display_names.get(str(model), str(model))
    return SIDE_BY_SIDE_LABEL_OVERRIDES.get(label, label)


side_by_side_all_models = list(
    dict.fromkeys(
        name
        for model_configs in SIDE_BY_SIDE_MODEL_GROUPS.values()
        for name in model_configs
    )
)
side_by_side_model_order = [
    name for name in SIDE_BY_SIDE_STYLE_ORDER if name in side_by_side_all_models
]
side_by_side_model_order.extend(
    name for name in side_by_side_all_models if name not in side_by_side_model_order
)

side_by_side_style_map = apply_seq_len_style_overrides(
    build_model_style_map(side_by_side_model_order)
)
side_by_side_style_map.update(
    {
        "DeltaNet_Comb_ST_Seq_Len_200-64K_lognormal_dataset_matched": ("X", "-.", "#D65A65"),
        "State_Passing_DeltaNet_Comb_ST": ("h", ":", "#FC9272"),
        "State_Weaving_DeltaNet_Comb_ST": ("P", "--", "#E34A33"),
        "mimetic:GLA_Comb_ST_Ref": ("<", "-", "#00441B"),
        "Bidirectional_GLA_Comb_ST_mean_output_mean_cache": ("8", ":", "#238B45"),
    }
)

fig = plt.figure(figsize=SIDE_BY_SIDE_FIGSIZE, dpi=400)
grid = fig.add_gridspec(2, 2, height_ratios=[4.0, 0.95], hspace=0.24, wspace=0.12)
axes = [fig.add_subplot(grid[0, 0]), fig.add_subplot(grid[0, 1], sharey=fig.axes[0])]
legend_axes = [fig.add_subplot(grid[1, 0]), fig.add_subplot(grid[1, 1])]
for legend_axis in legend_axes:
    legend_axis.axis("off")
fig.patch.set_facecolor("white")
fig.subplots_adjust(left=0.07, right=0.98, bottom=0.07, top=0.91)

legend_by_label = {}
label_panels = {}
direct_line_label_specs = []
for ax, (title, plot_df) in zip(axes, side_by_side_metric_dfs.items()):
    subset = plot_df[plot_df["metric"] == SIDE_BY_SIDE_METRIC]
    present_models = subset["model"].astype(str).unique().tolist()
    model_names = [name for name in side_by_side_model_order if name in present_models]
    model_names.extend(name for name in present_models if name not in model_names)

    ax.set_facecolor("#fbfbfb")
    x_values = subset["seqlen"].to_numpy(dtype=float, copy=False)
    positive_x_values = x_values[np.isfinite(x_values) & (x_values > 0.0)]

    for model in model_names:
        sub = subset[subset["model"] == model]
        if sub.empty:
            continue
        marker, linestyle, color = side_by_side_style_map.get(model, ("o", "-", None))
        label = side_by_side_label(model)
        agg = (
            sub.groupby("seqlen", observed=True)["value"]
            .agg(mean="mean", ci95=ci95_halfwidth)
            .reset_index()
            .sort_values("seqlen")
        )
        line = ax.plot(
            agg["seqlen"],
            agg["mean"],
            label=label,
            linestyle=linestyle,
            color=color,
            linewidth=2.15,
            marker=marker,
            markersize=5.0,
            alpha=0.96,
        )[0]
        err = agg["ci95"].fillna(0.0).to_numpy(dtype=float)
        if np.any(err > 0.0):
            mean_values = agg["mean"].to_numpy(dtype=float)
            ax.fill_between(
                agg["seqlen"],
                mean_values - err,
                mean_values + err,
                alpha=SIDE_BY_SIDE_CI_ALPHA,
                color=color,
            )
        if label in SIDE_BY_SIDE_DIRECT_LINE_LABELS:
            direct_line_label_specs.append((ax, label, agg.copy(), color))
        else:
            legend_by_label.setdefault(label, line)
            label_panels.setdefault(label, set()).add(title)

    ax.set_xscale("log")
    if positive_x_values.size:
        ax.set_xlim(float(positive_x_values.min()), float(positive_x_values.max()))
    ax.set_ylim(*SIDE_BY_SIDE_VALUE_YLIM)
    ax.set_xlabel("Number of In-context Samples")
    ax.set_title(title)
    ax.grid(True, which="major", ls="-", alpha=0.16)
    ax.grid(True, which="minor", ls="-", alpha=0.06)
    apply_pretraining_split_background(ax)


def add_direct_line_label(ax, label, agg, color, *, x_fraction=0.62):
    x_values = agg["seqlen"].to_numpy(dtype=float, copy=False)
    y_values = agg["mean"].to_numpy(dtype=float, copy=False)
    valid = np.isfinite(x_values) & np.isfinite(y_values) & (x_values > 0.0)
    if not np.any(valid):
        return
    x_values = x_values[valid]
    y_values = y_values[valid]
    order = np.argsort(x_values)
    x_values = x_values[order]
    y_values = y_values[order]
    x_min, x_max = ax.get_xlim()
    if ax.get_xscale() == "log":
        target_x = np.exp(np.log(x_min) + x_fraction * (np.log(x_max) - np.log(x_min)))
        nearest_idx = int(np.argmin(np.abs(np.log(x_values) - np.log(target_x))))
    else:
        target_x = x_min + x_fraction * (x_max - x_min)
        nearest_idx = int(np.argmin(np.abs(x_values - target_x)))
    ax.annotate(
        label,
        xy=(x_values[nearest_idx], y_values[nearest_idx]),
        xytext=(6, -1),
        textcoords="offset points",
        color=color,
        fontsize=8.2,
        fontweight="semibold",
        ha="left",
        va="top",
        bbox={"boxstyle": "round,pad=0.16", "facecolor": "white", "edgecolor": "none", "alpha": 0.72},
        clip_on=True,
    )


for label_ax, label, agg, color in direct_line_label_specs:
    add_direct_line_label(label_ax, label, agg, color)

axes[0].set_ylabel("Normalised ROC AUC")
axes[1].tick_params(axis="y", labelleft=False)
add_pretraining_split_legend(axes[0])

left_title, right_title = SIDE_BY_SIDE_MODEL_GROUPS.keys()
LEGEND_SPACER_LABEL = " "
SIDE_BY_SIDE_LEGEND_ORDERS = {
    left_title: [
        "Softmax\nNon-Causal",
        "Linear\nNon-Causal",
        "Linear Gated",
        # "Delta",
        "Softmax\nCausal",
        "Linear\nCausal",
        "Linear Gated: Subsampled 3K",
        # "Delta: Subsampled 3K",
    ],
    right_title: [
        "Linear Gated",
        "Linear Gated: State Weaving",
        "Linear Gated: Bidirectional",
        "Linear Gated: Mimetic",
        "Linear Gated: State Passing",
        # "Delta",
        # "Delta: State Weaving",
        # "Delta: Long Sequence",
        # "Delta: Bidirectional",
        # "Delta: State Passing",
    ],
}
SIDE_BY_SIDE_LEGEND_COLUMNS = {left_title: 3, right_title: 3}


def normalized_legend_label(label):
    return " ".join(str(label).split())


def ordered_legend_labels(labels, preferred_order):
    labels = list(labels)
    labels_by_normalized = {normalized_legend_label(label): label for label in labels}
    preferred = [
        labels_by_normalized[normalized_legend_label(label)]
        for label in preferred_order
        if normalized_legend_label(label) in labels_by_normalized
    ]
    preferred.extend(
        label
        for label in preferred_order
        if label == LEGEND_SPACER_LABEL
    )
    preferred_normalized = {
        normalized_legend_label(label)
        for label in preferred
        if label != LEGEND_SPACER_LABEL
    }
    return preferred + [
        label
        for label in labels
        if normalized_legend_label(label) not in preferred_normalized
    ]


def legend_input_order_for_visual_rows(labels, ncol):
    labels = list(labels)
    if ncol <= 1 or len(labels) <= ncol:
        return labels
    nrows = int(np.ceil(len(labels) / ncol))
    return [
        labels[row * ncol + col]
        for col in range(ncol)
        for row in range(nrows)
        if row * ncol + col < len(labels)
    ]


def legend_handles_for_labels(labels):
    return [
        mlines.Line2D([], [], alpha=0.0)
        if label == LEGEND_SPACER_LABEL
        else legend_by_label[label]
        for label in labels
    ]


legend_groups = [
    (
        legend_axis,
        legend_input_order_for_visual_rows(
            ordered_legend_labels(
                [label for label in legend_by_label if title in label_panels[label]],
                SIDE_BY_SIDE_LEGEND_ORDERS[title],
            ),
            SIDE_BY_SIDE_LEGEND_COLUMNS[title],
        ),
        SIDE_BY_SIDE_LEGEND_COLUMNS[title],
    )
    for legend_axis, title in zip(legend_axes, SIDE_BY_SIDE_MODEL_GROUPS)
]
for legend_axis, labels, ncol in legend_groups:
    if not labels:
        continue
    legend = legend_axis.legend(
        legend_handles_for_labels(labels),
        labels,
        loc="upper center",
        bbox_to_anchor=(0.5, 0.92),
        fontsize=8.2,
        frameon=True,
        facecolor="white",
        edgecolor="#d4d4d4",
        framealpha=1.0,
        handlelength=1.8,
        handletextpad=0.65,
        columnspacing=1.0,
        labelspacing=0.34,
        borderpad=0.42,
        ncol=ncol,
    )

side_by_side_paths = save_figure(fig, "seq_len_normalized_global_side_by_side")
display(fig)
print("Saved side-by-side figures:")
for path in side_by_side_paths:
    print(f" - {path}")


In [ ]:
from pfns.experiments.model_benchmarks.analysis import (
    add_normalized_comparison_metrics,
    compute_global_normalization_constants,
)
from pfns.experiments.model_benchmarks.comparison_analysis import ci95_halfwidth
from pfns.experiments.model_benchmarks.plotting import (
    PLOT_FACE_COLOR,
    apply_pretraining_split_background,
)
import matplotlib.lines as mlines
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import seaborn as sns

DEBUG_PERFORMANCE_MODEL_GROUPS = {
    "Causal variants": [
        "Linear_Attention_Comb_ST",
        "Linear_Attention_Comb_ST_fro_norm",
    ],
    "Non-causal variants": [
        "Linear_Attention_Non_Causal",
        "Linear_Attention_Non_Causal_fro_norm",
    ],
}
DEBUG_PERFORMANCE_VARIANT_STYLES = {
    "baseline": {"label": "Standard", "color": "#00796B", "marker": "o"},
    "fro_norm": {"label": "Frobenius norm", "color": "#D55E00", "marker": "s"},
}
DEBUG_PERFORMANCE_MODEL_VARIANTS = {
    "Linear_Attention_Comb_ST": "baseline",
    "Linear_Attention_Comb_ST_fro_norm": "fro_norm",
    "Linear_Attention_Non_Causal": "baseline",
    "Linear_Attention_Non_Causal_fro_norm": "fro_norm",
}
DEBUG_PERFORMANCE_RAW_METRICS = ("roc_auc", "acc")
DEBUG_PERFORMANCE_METRIC_STYLES = {
    "normalized_global_roc_auc": {"label": "Global norm. ROC AUC", "linestyle": "-", "axis": "left"},
    "normalized_global_acc": {"label": "Global norm. Accuracy", "linestyle": "--", "axis": "right"},
}
DEBUG_PERFORMANCE_FIGSIZE = (12.4, 4.35)
DEBUG_PERFORMANCE_CI_ALPHA = 0.09
DEBUG_PERFORMANCE_LINE_ALPHA = 0.96
DEBUG_PERFORMANCE_LINE_WIDTH = 2.25
DEBUG_PERFORMANCE_MARKER_SIZE = 5.4
DEBUG_PERFORMANCE_Y_PAD_FRAC = 0.16
DEBUG_PERFORMANCE_MIN_Y_PAD = 0.01
DEBUG_PERFORMANCE_Y_LIMIT_OVERRIDES = {
    # Use e.g. "normalized_global_roc_auc": (0.45, 0.85) for a fixed paper crop.
    "normalized_global_roc_auc": (0.25, 0.65),
    "normalized_global_acc": (0.22, 0.55),
}

if "metric_analysis_df" not in globals() or metric_analysis_df.empty:
    raise RuntimeError("No metric_analysis_df found. Run the benchmark loading and plotting setup cells first.")

required_models = [
    model
    for group_models in DEBUG_PERFORMANCE_MODEL_GROUPS.values()
    for model in group_models
]
performance_base_df = metric_analysis_df[
    metric_analysis_df["model"].astype(str).isin(required_models)
    & metric_analysis_df["metric"].astype(str).isin(DEBUG_PERFORMANCE_RAW_METRICS)
].copy()

missing_models = sorted(set(required_models) - set(performance_base_df["model"].astype(str).unique()))
missing_metrics = sorted(
    set(DEBUG_PERFORMANCE_RAW_METRICS)
    - set(performance_base_df["metric"].astype(str).unique())
)
if missing_models:
    print(f"Missing requested models in metric_analysis_df: {missing_models}")
if missing_metrics:
    print(f"Missing requested metrics in metric_analysis_df: {missing_metrics}")
if performance_base_df.empty:
    raise RuntimeError("No rows found for the requested hidden-state debug performance models.")


def _global_normalization_constants_for_debug_metrics():
    constants = dict(globals().get("GLOBAL_NORMALIZATION_CONSTANTS", {}))
    missing = [metric for metric in DEBUG_PERFORMANCE_RAW_METRICS if metric not in constants]
    if missing:
        reference_df = globals().get("global_normalization_reference_df", metric_analysis_df)
        constants.update(
            compute_global_normalization_constants(
                reference_df,
                metric_keys=DEBUG_PERFORMANCE_RAW_METRICS,
                higher_is_better_metrics=HIGHER_IS_BETTER_METRICS,
                group_cols=("rep",),
                lower_bound_reference_max_x=GLOBAL_NORMALIZATION_LOWER_BOUND_REFERENCE_MAX_SEQLEN,
            )
        )
    still_missing = [metric for metric in DEBUG_PERFORMANCE_RAW_METRICS if metric not in constants]
    if still_missing:
        raise RuntimeError(f"Could not compute global normalization constants for: {still_missing}")
    return constants


performance_plot_df = add_normalized_comparison_metrics(
    performance_base_df,
    metric_keys=DEBUG_PERFORMANCE_RAW_METRICS,
    higher_is_better_metrics=HIGHER_IS_BETTER_METRICS,
    group_cols=("rep",),
    normalization_scope="group",
    normalization_constants=_global_normalization_constants_for_debug_metrics(),
    require_normalization_constants=True,
    normalized_prefix="normalized_global_",
)
performance_plot_df = performance_plot_df[
    performance_plot_df["metric"].astype(str).isin(DEBUG_PERFORMANCE_METRIC_STYLES)
].copy()
if performance_plot_df.empty:
    raise RuntimeError("No globally normalized performance rows were produced.")


def _aggregate_metric(df, model, metric):
    sub = df[(df["model"].astype(str) == model) & (df["metric"].astype(str) == metric)]
    if sub.empty:
        return sub
    return (
        sub.groupby("seqlen", observed=True)["value"]
        .agg(mean="mean", ci95=ci95_halfwidth)
        .reset_index()
        .sort_values("seqlen")
    )


agg_by_model_metric = {
    (model, metric): _aggregate_metric(performance_plot_df, model, metric)
    for model in required_models
    for metric in DEBUG_PERFORMANCE_METRIC_STYLES
}


def _metric_limits(metric):
    override = DEBUG_PERFORMANCE_Y_LIMIT_OVERRIDES.get(metric)
    if override is not None:
        return override
    values = []
    for (model, metric_key), agg in agg_by_model_metric.items():
        if metric_key != metric or agg.empty:
            continue
        mean = agg["mean"].to_numpy(dtype=float)
        err = agg["ci95"].fillna(0.0).to_numpy(dtype=float)
        values.extend((mean - err).tolist())
        values.extend((mean + err).tolist())
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if values.size == 0:
        return None
    low = float(values.min())
    high = float(values.max())
    pad = max((high - low) * DEBUG_PERFORMANCE_Y_PAD_FRAC, DEBUG_PERFORMANCE_MIN_Y_PAD)
    return max(0.0, low - pad), min(1.0, high + pad)


metric_ylims = {
    metric: _metric_limits(metric)
    for metric in DEBUG_PERFORMANCE_METRIC_STYLES
}
positive_x_values = performance_plot_df["seqlen"].to_numpy(dtype=float, copy=False)
positive_x_values = positive_x_values[np.isfinite(positive_x_values) & (positive_x_values > 0.0)]

sns.set_theme(style="whitegrid", font_scale=1)
fig, axes = plt.subplots(1, 2, figsize=DEBUG_PERFORMANCE_FIGSIZE, dpi=400, sharex=True)
fig.patch.set_facecolor("white")
fig.subplots_adjust(left=0.06, bottom=0.12, right=0.98, top=0.92, wspace=0.45)

for panel_idx, (ax_left, (panel_title, panel_models)) in enumerate(zip(axes, DEBUG_PERFORMANCE_MODEL_GROUPS.items())):
    ax_right = ax_left.twinx()
    ax_left.set_facecolor(PLOT_FACE_COLOR)
    ax_right.patch.set_visible(False)

    for metric, metric_style in DEBUG_PERFORMANCE_METRIC_STYLES.items():
        metric_axis = ax_left if metric_style["axis"] == "left" else ax_right
        for model in panel_models:
            agg = agg_by_model_metric[(model, metric)]
            if agg.empty:
                continue
            variant_style = DEBUG_PERFORMANCE_VARIANT_STYLES[DEBUG_PERFORMANCE_MODEL_VARIANTS[model]]
            metric_axis.plot(
                agg["seqlen"],
                agg["mean"],
                color=variant_style["color"],
                marker=variant_style["marker"],
                linestyle=metric_style["linestyle"],
                linewidth=DEBUG_PERFORMANCE_LINE_WIDTH,
                markersize=DEBUG_PERFORMANCE_MARKER_SIZE,
                markeredgewidth=0.0,
                alpha=DEBUG_PERFORMANCE_LINE_ALPHA,
            )
            err = agg["ci95"].fillna(0.0).to_numpy(dtype=float)
            if np.any(err > 0.0):
                mean_values = agg["mean"].to_numpy(dtype=float)
                metric_axis.fill_between(
                    agg["seqlen"],
                    mean_values - err,
                    mean_values + err,
                    color=variant_style["color"],
                    alpha=DEBUG_PERFORMANCE_CI_ALPHA,
                    linewidth=0,
                )

    ax_left.set_xscale("log")
    if positive_x_values.size:
        ax_left.set_xlim(float(positive_x_values.min()), float(positive_x_values.max()))
    for metric, ylim in metric_ylims.items():
        if ylim is None:
            continue
        target_axis = ax_left if DEBUG_PERFORMANCE_METRIC_STYLES[metric]["axis"] == "left" else ax_right
        target_axis.set_ylim(*ylim)
        target_axis.yaxis.set_major_locator(mticker.MaxNLocator(nbins=5))
        target_axis.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.2f"))

    apply_pretraining_split_background(ax_left)
    ax_left.set_title(panel_title)
    ax_left.set_xlabel("Number of In-context Samples")
    ax_left.set_ylabel("Global normalised ROC AUC" if panel_idx == 0 else "")
    ax_right.set_ylabel("Global normalised Accuracy" if panel_idx == len(axes) - 1 else "")
    if panel_idx > 0:
        ax_left.tick_params(axis="y", left=False, labelleft=False)
        ax_left.spines["left"].set_visible(False)
    if panel_idx < len(axes) - 1:
        ax_right.tick_params(axis="y", right=False, labelright=False)
        ax_right.spines["right"].set_visible(False)
    ax_left.set_axisbelow(True)
    ax_left.grid(True, which="major", ls="-", color="#b8b8b8", alpha=0.34, linewidth=0.75)
    ax_left.grid(False, which="minor")
    ax_right.grid(False)
    for spine in (*ax_left.spines.values(), *ax_right.spines.values()):
        spine.set_color("#d7d7d7")
        spine.set_linewidth(0.9)

variant_handles = [
    mlines.Line2D(
        [], [],
        color=style["color"],
        marker=style["marker"],
        linestyle="-",
        linewidth=DEBUG_PERFORMANCE_LINE_WIDTH,
        markersize=DEBUG_PERFORMANCE_MARKER_SIZE,
        markeredgewidth=0.0,
        label=style["label"],
    )
    for style in DEBUG_PERFORMANCE_VARIANT_STYLES.values()
]
metric_handles = [
    mlines.Line2D(
        [], [],
        color="#222222",
        linestyle=metric_style["linestyle"],
        linewidth=DEBUG_PERFORMANCE_LINE_WIDTH,
        label=metric_style["label"],
    )
    for metric_style in DEBUG_PERFORMANCE_METRIC_STYLES.values()
]
fig.legend(
    handles=variant_handles + metric_handles,
    loc="lower center",
    bbox_to_anchor=(0.5, 0.045),
    ncol=4,
    fontsize=plt.rcParams["font.size"],
    frameon=False,
    handlelength=2.1,
    columnspacing=1.45,
)
fig.tight_layout(rect=(0, 0.12, 1, 1), w_pad=0.35)
fig.subplots_adjust(wspace=0.04)

debug_performance_paths = save_figure(fig, "seq_len_hidden_state_debug_performance_global_normalized")
display(fig)
print("Saved split hidden-state debug performance figures:")
for path in debug_performance_paths:
    print(f" - {path}")


# Tabels with the detailed results for each model and sequence length.

## Performance, Timing and Memory Tables

Summary tables by model, by sequence-length bucket, and by exact sequence length.


In [ ]:
for panel_name, df, kwargs in TABLE_PANEL_CONFIGS:
    display_panel_tables(df, panel_name=panel_name, **kwargs)


## Performance rank table

In [ ]:
overall_ranks = rank_tables["overall_ranks"]
bucket_ranks = rank_tables["bucket_ranks"]
x_ranks = rank_tables["x_ranks"]

for metric in sorted(metric_plot_df["metric"].unique()):
    print(f"=== Mean rank for {metric} (lower is better) ===")
    overall = (
        overall_ranks[overall_ranks["metric"] == metric]
        .sort_values("rank")
        .drop(columns="metric")
        .reset_index(drop=True)
    )
    display(overall.style.format({"rank": "{:.2f}"}).background_gradient(subset=["rank"], cmap="Greens_r"))

print("=== Mean rank by bucket ===")
_display_pivot_by_metric(
    bucket_ranks,
    panel_name="Rank by bucket",
    pivot_col="bucket",
    value_col="rank",
    value_fmt="{:.2f}",
    cmap="Greens_r",
)

print("=== Mean rank by sequence length ===")
_display_pivot_by_metric(
    x_ranks,
    panel_name="Rank by sequence length",
    pivot_col="seqlen",
    value_col="rank",
    value_fmt="{:.2f}",
    cmap="Greens_r",
)

## Setting Comparison Across Sequence-Length Generalisation

Paired gain and Wilcoxon-Holm significance tests for canonical settings (`Comb_MT`, `Comb_ST`, `Int_ST`, `Int_MT`) across all lengths, pre-training range, and generalisation range.


In [ ]:
from pfns.experiments.model_benchmarks.comparison_analysis import run_comparison_analysis
from pfns.experiments.model_benchmarks.setting_analysis import get_setting_preprocess

SEQ_METRIC_DIRECTION = {
    "acc": True,
    "roc_auc": True,
    "ce": False,
}

SEQ_METRIC_LABELS = {
    "acc": "Accuracy",
    "roc_auc": "ROC AUC",
    "ce": "Cross Entropy",
}

SETTING_GENERALIZATION_ANALYSIS = {
    "metric": "roc_auc",  # one of: acc, roc_auc, ce
    "target_labels": ["Comb_MT", "Comb_ST", "Int_ST", "Int_MT"],
    "reference_label": "Int_MT",
    "error_bars": "ci95",  # std or ci95
    "pair_by_rep": True,
    "seqlen_cutoff": 1_000,
    "wilcoxon_alpha": 0.05,
}

if "metric_analysis_df" not in globals() or metric_analysis_df.empty:
    raise RuntimeError("No metric_analysis_df found. Run the plotting/configuration cell first.")

metric = SETTING_GENERALIZATION_ANALYSIS["metric"]
if metric not in SEQ_METRIC_DIRECTION:
    raise ValueError(f"Unsupported metric {metric!r}. Choose from {list(SEQ_METRIC_DIRECTION)}")

metric_slice = metric_analysis_df[metric_analysis_df["metric"] == metric].copy()
if metric_slice.empty:
    raise RuntimeError(f"No rows found for metric {metric!r} in metric_analysis_df.")

target_labels = list(dict.fromkeys(SETTING_GENERALIZATION_ANALYSIS["target_labels"]))
try:
    pre = get_setting_preprocess(results_df=metric_slice, target_settings=target_labels)
except RuntimeError as exc:
    print(f"Skipping setting comparison: {exc}")
else:
    comparison_base = pre["filtered_results"].copy()
    presence = pre["presence"].reindex(columns=target_labels, fill_value=False).sort_index()
    eligible_model_types = pre["eligible_model_types"]

    print("Canonical setting availability by model type:")
    display(presence)
    print("Eligible model types (all requested settings available):", eligible_model_types)

    cutoff = int(SETTING_GENERALIZATION_ANALYSIS["seqlen_cutoff"])
    range_slices = [
        ("all_lengths", comparison_base),
        (f"pretraining_le_{cutoff}", comparison_base[comparison_base["seqlen"] <= cutoff].copy()),
        (f"generalization_gt_{cutoff}", comparison_base[comparison_base["seqlen"] > cutoff].copy()),
    ]

    higher_is_better = SEQ_METRIC_DIRECTION[metric]
    pair_cols = ["model_type", "seqlen", "rep"] if SETTING_GENERALIZATION_ANALYSIS["pair_by_rep"] else ["model_type", "seqlen"]

    for range_name, range_df in range_slices:
        print(f"\n=== Setting comparison: {range_name} ===")
        if range_df.empty:
            print("Skipping: no rows in this range.")
            continue

        available_settings = [
            label for label in target_labels if label in set(range_df["setting"].astype(str).unique())
        ]
        if len(available_settings) < 2:
            print(f"Skipping: need at least two settings in range, found {available_settings}.")
            continue

        reference_label = SETTING_GENERALIZATION_ANALYSIS["reference_label"]
        if reference_label not in available_settings:
            label_means = (
                range_df.groupby("setting", observed=True)["value"].mean().reindex(available_settings)
            )
            reference_label = label_means.idxmax() if higher_is_better else label_means.idxmin()

        try:
            analysis = run_comparison_analysis(
                comparison_results=range_df,
                metric_col="value",
                metric_label=SEQ_METRIC_LABELS[metric],
                compare_col="setting",
                target_labels=available_settings,
                pair_cols=pair_cols,
                higher_better=higher_is_better,
                reference_label=reference_label,
                unit="seqlen",
                error_bars=SETTING_GENERALIZATION_ANALYSIS["error_bars"],
                comparison_label="setting",
                include_boxplot=True,
                include_pairwise_tables=True,
                include_cd_diagram=True,
                wilcoxon_alpha=SETTING_GENERALIZATION_ANALYSIS["wilcoxon_alpha"],
                empty_error_message=(
                    "No complete paired rows found across requested settings for this length range. "
                    "Try pair_by_rep=True, or evaluate more model types."
                ),
            )
        except RuntimeError as exc:
            print(f"Skipping {range_name}: {exc}")
            continue

        wilcoxon_summary = analysis["wilcoxon"]
        n_pairs = wilcoxon_summary["n_pairs"]

        print(f"Metric: {SEQ_METRIC_LABELS[metric]} ({metric})")
        print(f"Reference setting: {analysis['gain']['reference_label']}")
        print(f"Compared settings: {analysis['target_labels']}")
        print(f"Pairing columns: {pair_cols}")
        print(f"Complete paired rows used: {analysis['n_complete_pairs']}")
        print(f"Wilcoxon/Holm paired rows used: {int(n_pairs)}")

        print("Setting means:")
        display(analysis["gain"]["label_means"].to_frame("mean"))

        print("\nPaired gain summary by setting (positive = better than reference):")
        display(
            analysis["gain"]["gain_summary"].style.format(
                {
                    "mean_gain": "{:+.5f}",
                    "std_gain": "{:.5f}",
                    "sem_gain": "{:.5f}",
                    "ci95": "{:.5f}",
                    "ci95_low": "{:+.5f}",
                    "ci95_high": "{:+.5f}",
                    "n_pairs": "{:.0f}",
                    "share_pairs_better": "{:.1%}",
                }
            ).background_gradient(subset=["mean_gain"], cmap="RdYlGn")
        )

        print("\nPairwise Wilcoxon p-values with Holm significance flag:")
        p_value_table = wilcoxon_summary["p_value_table"]
        display(
            p_value_table.style.format({"p_raw": "{:.6f}"}).apply(
                lambda col: ["background-color: #c7f9cc" if bool(v) else "" for v in col],
                subset=["significant_holm"],
            )
        )

        print("Average ranks used in the CD diagram (1 = best):")
        rank_table = wilcoxon_summary["rank_table"]
        display(rank_table.sort_values("average_rank").style.format({"average_rank": "{:.3f}"}))

        if analysis["pairwise"] is not None:
            print("Pairwise mean gain matrix (row vs column):")
            display(analysis["pairwise"]["pairwise_mean"].style.format("{:+.4f}").background_gradient(cmap="RdYlGn", axis=None))
            print("Pairwise 95% CI half-width matrix:")
            display(analysis["pairwise"]["pairwise_ci95"].style.format("{:.4f}").background_gradient(cmap="Blues", axis=None))
            print("Pairwise significance proxy (95% CI excludes zero):")
            display(analysis["pairwise"]["pairwise_sig"])

        for fig_key in ["gain_barh", "gain_boxplot", "wilcoxon_cd"]:
            if fig_key in analysis["figures"]:
                fig, ax = analysis["figures"][fig_key]
                if fig_key == "wilcoxon_cd":
                    ax.set_title(
                        f"Wilcoxon/Holm comparison diagram ({SEQ_METRIC_LABELS[metric]}, unit=seqlen, range={range_name})",
                        y=0.98,
                    )
                display(fig)
                plt.close(fig)

## Equal-Parameter Comparison Across Sequence-Length Generalisation

Paired gain and Wilcoxon-Holm significance tests for registry-defined equal-parameter models across all lengths, pre-training range, and generalisation range.


In [ ]:
import matplotlib.pyplot as plt
from pfns.experiments.model_benchmarks.model_registry import get_models_from_families
from pfns.experiments.model_benchmarks.comparison_analysis import run_comparison_analysis

if "SEQ_METRIC_DIRECTION" not in globals() or "SEQ_METRIC_LABELS" not in globals():
    raise RuntimeError(
        "Missing shared metric metadata. Run the 'Setting Comparison Across Sequence-Length Generalisation' code cell first."
    )

EQUAL_PARAMS_GENERALIZATION_ANALYSIS = {
    "metric": "roc_auc",  # one of: acc, roc_auc, ce
    "unit": "seqlen",
    "reference_label": None,  # auto-select best mean in each range
    "error_bars": "ci95",
    "pair_by_rep": True,
    "seqlen_cutoff": 1_000,
    "wilcoxon_alpha": 0.05,
}

if "metric_analysis_df" not in globals() or metric_analysis_df.empty:
    raise RuntimeError("No metric_analysis_df found. Run the plotting/configuration cell first.")

metric = EQUAL_PARAMS_GENERALIZATION_ANALYSIS["metric"]
if metric not in SEQ_METRIC_DIRECTION:
    raise ValueError(f"Unsupported metric {metric!r}. Choose from {list(SEQ_METRIC_DIRECTION)}")

metric_slice = metric_analysis_df[metric_analysis_df["metric"] == metric].copy()
if metric_slice.empty:
    raise RuntimeError(f"No rows found for metric {metric!r} in metric_analysis_df.")

registered_equal_models = list(get_models_from_families(["equal_params"]).keys())
available_models = set(metric_slice["model"].astype(str).unique())
target_models = [m for m in registered_equal_models if m in available_models]

print(f"Equal-params models available in this run: {target_models}")
if len(target_models) < 2:
    print("Skipping equal-params comparison: need at least two equal-params models in metric_analysis_df.")
else:
    comparison_base = metric_slice[metric_slice["model"].isin(target_models)].copy()
    cutoff = int(EQUAL_PARAMS_GENERALIZATION_ANALYSIS["seqlen_cutoff"])
    range_slices = [
        ("all_lengths", comparison_base),
        (f"pretraining_le_{cutoff}", comparison_base[comparison_base["seqlen"] <= cutoff].copy()),
        (f"generalization_gt_{cutoff}", comparison_base[comparison_base["seqlen"] > cutoff].copy()),
    ]

    higher_is_better = SEQ_METRIC_DIRECTION[metric]
    pair_cols = ["seqlen", "rep"] if EQUAL_PARAMS_GENERALIZATION_ANALYSIS["pair_by_rep"] else ["seqlen"]

    for range_name, range_df in range_slices:
        print(f"\n=== Equal-params comparison: {range_name} ===")
        if range_df.empty:
            print("Skipping: no rows in this range.")
            continue

        available_in_range = [
            m for m in target_models if m in set(range_df["model"].astype(str).unique())
        ]
        if len(available_in_range) < 2:
            print(f"Skipping: need at least two models in range, found {available_in_range}.")
            continue

        model_means = range_df.groupby("model", observed=True)["value"].mean().reindex(available_in_range)
        reference_label = EQUAL_PARAMS_GENERALIZATION_ANALYSIS["reference_label"]
        if reference_label is None or reference_label not in available_in_range:
            reference_label = model_means.idxmax() if higher_is_better else model_means.idxmin()

        try:
            analysis = run_comparison_analysis(
                comparison_results=range_df,
                metric_col="value",
                metric_label=SEQ_METRIC_LABELS[metric],
                compare_col="model",
                target_labels=available_in_range,
                pair_cols=pair_cols,
                higher_better=higher_is_better,
                reference_label=reference_label,
                unit=EQUAL_PARAMS_GENERALIZATION_ANALYSIS["unit"],
                error_bars=EQUAL_PARAMS_GENERALIZATION_ANALYSIS["error_bars"],
                comparison_label="model",
                include_boxplot=False,
                include_pairwise_tables=True,
                include_cd_diagram=True,
                wilcoxon_alpha=EQUAL_PARAMS_GENERALIZATION_ANALYSIS["wilcoxon_alpha"],
                empty_error_message=(
                    "No complete paired rows found across equal-params models for this length range. "
                    "Try pair_by_rep=True, or evaluate more equal-params models."
                ),
            )
        except RuntimeError as exc:
            print(f"Skipping {range_name}: {exc}")
            continue

        wilcoxon_summary = analysis["wilcoxon"]
        n_pairs = wilcoxon_summary["n_pairs"]

        print(f"Metric: {SEQ_METRIC_LABELS[metric]} ({metric})")
        print(f"Reference model: {analysis['gain']['reference_label']}")
        print(f"Compared models: {analysis['target_labels']}")
        print(f"Pairing columns: {pair_cols}")
        print(f"Complete paired rows used: {analysis['n_complete_pairs']}")
        print(f"Wilcoxon/Holm paired rows used: {int(n_pairs)}")

        print("Equal-params model means:")
        display(model_means.sort_values(ascending=not higher_is_better).to_frame("mean"))

        print("\nPaired gain summary by model (positive = better than reference):")
        display(
            analysis["gain"]["gain_summary"].style.format(
                {
                    "mean_gain": "{:+.5f}",
                    "std_gain": "{:.5f}",
                    "sem_gain": "{:.5f}",
                    "ci95": "{:.5f}",
                    "ci95_low": "{:+.5f}",
                    "ci95_high": "{:+.5f}",
                    "n_pairs": "{:.0f}",
                    "share_pairs_better": "{:.1%}",
                }
            ).background_gradient(subset=["mean_gain"], cmap="RdYlGn")
        )

        print("\nPairwise Wilcoxon p-values with Holm significance flag:")
        p_value_table = wilcoxon_summary["p_value_table"]
        display(
            p_value_table.style.format({"p_raw": "{:.6f}"}).apply(
                lambda col: ["background-color: #c7f9cc" if bool(v) else "" for v in col],
                subset=["significant_holm"],
            )
        )

        print("Average ranks used in the CD diagram (1 = best):")
        rank_table = wilcoxon_summary["rank_table"]
        display(rank_table.sort_values("average_rank").style.format({"average_rank": "{:.3f}"}))

        if analysis["pairwise"] is not None:
            print("Pairwise mean gain matrix (row vs column):")
            display(analysis["pairwise"]["pairwise_mean"].style.format("{:+.4f}").background_gradient(cmap="RdYlGn", axis=None))
            print("Pairwise 95% CI half-width matrix:")
            display(analysis["pairwise"]["pairwise_ci95"].style.format("{:.4f}").background_gradient(cmap="Blues", axis=None))
            print("Pairwise significance proxy (95% CI excludes zero):")
            display(analysis["pairwise"]["pairwise_sig"])

        for fig_key in ["gain_barh", "wilcoxon_cd"]:
            if fig_key in analysis["figures"]:
                fig, ax = analysis["figures"][fig_key]
                if fig_key == "wilcoxon_cd":
                    ax.set_title(
                        f"Wilcoxon/Holm comparison diagram ({SEQ_METRIC_LABELS[metric]}, unit={EQUAL_PARAMS_GENERALIZATION_ANALYSIS['unit']}, range={range_name})",
                        y=0.98,
                    )
                display(fig)
                plt.close(fig)

## Setup-Averaged Sequence-Length Curves

Average performance, latency, and memory curves across all models that share the same canonical setup (`Comb_MT`, `Comb_ST`, `Int_ST`, `Int_MT`).


In [ ]:
from pfns.experiments.model_benchmarks.setting_analysis import (
    CANONICAL_SETUP_DISPLAY_NAMES,
    build_setup_averaged_df,
)

setup_metric_df = build_setup_averaged_df(metric_analysis_df)
setup_timing_df = build_setup_averaged_df(timing_analysis_df)
setup_memory_df = build_setup_averaged_df(memory_analysis_df)

if setup_metric_df is None or setup_metric_df.empty:
    print("No canonical setup labels found in loaded models; skipping setup-averaged plots.")
else:
    setup_metric_plot_df = build_normalized_metric_plot_df(setup_metric_df)

    setup_order = [
        name
        for name in CANONICAL_SETUP_DISPLAY_NAMES.values()
        if name in set(setup_metric_df["model"].unique())
    ]
    setup_style_map = build_model_style_map(setup_order)

    setup_plot_panel_configs = [
        ("metric", setup_metric_plot_df, {"log_x": True, "error_bars": "ci95", "error_style": "band", "title_suffix": " (Setup average) vs Number of In-context Samples", "figsize": PAPER_PLOT_FIGSIZE}),
        *[
            (
                panel_name,
                setup_metric_plot_df,
                {
                    "log_x": True,
                    "error_bars": "ci95",
                    "error_style": "band",
                    "title_suffix": f" Setup Average: {config['title']}",
                    "title_override": "" if panel_name == "metric_normalized_global" else None,
                    "y_label": "Normalised ROC AUC" if panel_name == "metric_normalized_global" else None,
                    "figsize": PAPER_PLOT_FIGSIZE,
                },
            )
            for panel_name, config in NORMALIZATION_VIEWS.items()
        ],
        ("timing", setup_timing_df, {"show_std": True, "log_x": True, "log_y_metrics": ["fit_time_ms"], "title_suffix": " (Setup average) vs Number of In-context Samples", "figsize": PAPER_PLOT_FIGSIZE}),
        ("memory", setup_memory_df, {"log_x": False, "title_suffix": " (Setup average) vs Number of In-context Samples", "figsize": PAPER_PLOT_FIGSIZE}),
    ]

    setup_saved_figure_paths = []
    for panel_name, df, opts in setup_plot_panel_configs:
        fig, axes = plot_panel(
            df,
            PLOT_SPECS["metric_raw"] if panel_name == "metric" else PLOT_SPECS[panel_name],
            style_map_override=setup_style_map,
            show=False,
            **opts,
        )
        setup_saved_figure_paths.extend(save_figure(fig, f"seq_len_setup_{panel_name}"))
        if fig is not None:
            display(fig)

    setup_rank_tables = compute_mean_rank_tables(setup_metric_df, x_col="seqlen")
    fig, axes = plot_panel(
        setup_rank_tables["x_ranks"],
        PLOT_SPECS["metric_raw"],
        value_col="rank",
        log_x=True,
        invert_y=True,
        title_suffix=" Rank (1 is best, setup average)",
        style_map_override=setup_style_map,
        figsize=PAPER_PLOT_FIGSIZE,
        show=False,
    )
    setup_saved_figure_paths.extend(save_figure(fig, "seq_len_setup_metric_rank"))
    if fig is not None:
        display(fig)

    print("Saved setup-averaged figures:")
    for path in setup_saved_figure_paths:
        print(f" - {path}")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def sample_shifted_capped_lognormal(
    n=200_000,
    *,
    min_len=200,
    max_len=64_000,
    target_mean=1000,
    sigma=1.2,
    seed=13,
):
    rng = np.random.default_rng(seed)

    # Solve mu so clipped sample mean matches target_mean.
    z = rng.standard_normal(n)

    def sample_for_mu(mu):
        return np.minimum(min_len + np.exp(mu + sigma * z), max_len)

    lo, hi = np.log(1), np.log(max_len)
    for _ in range(80):
        mid = (lo + hi) / 2
        if sample_for_mu(mid).mean() < target_mean:
            lo = mid
        else:
            hi = mid

    mu = (lo + hi) / 2
    seq_lens = np.rint(sample_for_mu(mu)).astype(int)
    return seq_lens, {"mu": mu, "sigma": sigma}
for sigma in [1.2, 1.4, 1.6, 1.8, 2.0]:
    seq_lens, _ = sample_shifted_capped_lognormal(sigma=sigma, seed=13)
    print(
        f"sigma={sigma:.1f}",
        f"mean={seq_lens.mean():.0f}",
        f"median={np.median(seq_lens):.0f}",
        f"p99={np.quantile(seq_lens, 0.99):.0f}",
        f"p999={np.quantile(seq_lens, 0.999):.0f}",
        f"max={seq_lens.max()}",
        f">10K={(seq_lens > 10_000).mean():.3%}",
    )


fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(seq_lens[seq_lens <= 2000], bins=np.arange(200, 2025, 25))
axes[0].set_title("Body: shifted lognormal")
axes[0].set_xlabel("seq_len")
axes[0].set_ylabel("count")

axes[1].hist(seq_lens, bins=np.geomspace(200, 64000, 100), density=True, alpha=0.85)
axes[1].set_xscale("log")
axes[1].axvline(seq_lens.mean(), color="C1", label=f"mean={seq_lens.mean():.0f}")
axes[1].axvline(1000, color="black", linestyle=":", label="1K")
axes[1].axvline(10000, color="gray", linestyle=":", label="10K")
axes[1].legend()
plt.tight_layout()
plt.show()
